# Capítulo 5 · Actividad: ¿cuánto consume este auto?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riosinda/ml-course/blob/main/notebooks/exercises/chapter_05/02_actividad_mpg_regresion.ipynb)

Entrena una **regresión** que prediga el rendimiento de un auto en **millas por galón** (`mpg`) y consigue el mejor modelo que puedas. Haz todo el proceso que vimos en clase; el preprocesamiento lo decides tú.

| Variable | Significado |
|---|---|
| `mpg` | **Target**: millas por galón (más alto = consume menos) |
| `cylinders` | Número de cilindros |
| `displacement` | Cilindrada del motor (pulgadas cúbicas) |
| `horsepower` | Potencia (caballos de fuerza) |
| `weight` | Peso (libras) |
| `acceleration` | Segundos para ir de 0 a 60 mph |
| `model_year` | Año del modelo (70 = 1970) |
| `origin` | Región de origen: `usa`, `europe` o `japan` |
| `name` | Nombre del auto |

Como mínimo, tu notebook debe incluir:

- Un **modelo lineal base**: coeficientes, métricas de error y $R^2$.
- Los mismos modelos que vimos en clase: regresión simple, lineal múltiple, Ridge, Lasso, polinomial y RANSAC.
- Una **tabla final** comparando los modelos con validación cruzada y test, y tus **conclusiones**: ¿qué modelo elegirías y por qué?

1. **Copia esta notebook a `workspace/`** y trabaja sobre la copia.
2. Si todavía no tienes los datos: `uv run python scripts/download_data.py`

Seguimos el mismo orden de trabajo del Titanic: separar train y test antes de decidir nada, tomar las decisiones con validación cruzada solo en train, y usar el test al final. Además de entrenar los modelos de la clase, calculamos a mano qué hace una regresión lineal (sección 4) y lo comparamos con scikit-learn.

Las métricas que usamos, con `mpg` como unidad:

- **RMSE**: el error típico de la predicción, en mpg. Castiga más los errores grandes. Más bajo es mejor.
- **MAE**: el error promedio sin signo, en mpg.
- **MAPE**: el error promedio como porcentaje del valor real.
- **R²**: la proporción de la variación de `mpg` que explica el modelo. 1 es perfecto y 0 es igual a predecir siempre el promedio.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn import set_config
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, LinearRegression, RANSACRegressor, Ridge
from sklearn.metrics import (mean_absolute_error, mean_absolute_percentage_error, mean_squared_error,
                             r2_score, root_mean_squared_error)
from sklearn.model_selection import GridSearchCV, KFold, RepeatedKFold, cross_val_score, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, PolynomialFeatures, StandardScaler

# Que scikit-learn devuelva DataFrames (con nombres de columna)
set_config(transform_output="pandas")
sns.set_theme(style="whitegrid")

SEED = 42
np.random.seed(SEED)

# Carpeta data/ en la raíz del repo: subimos de carpeta en carpeta hasta dar con pyproject.toml
raiz = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
DATA_DIR = raiz / "data"

df = pd.read_csv(DATA_DIR / "mpg.csv")
df.head()

## 1. Exploración de los datos

Primero miramos cómo vienen los datos. En esta sección solo describimos: las decisiones se toman más adelante, con train.

In [ ]:
print("Filas y columnas:", df.shape)
df.info()

In [ ]:
df.describe().round(2)

In [ ]:
print("Filas duplicadas:", df.duplicated().sum())
print("Nombres distintos:", df["name"].nunique())
print("\nAutos por origen:")
print(df["origin"].value_counts())
print("\nAutos por número de cilindros:")
print(df["cylinders"].value_counts().sort_index())
print("\nAutos con horsepower faltante:")
display(df[df["horsepower"].isna()])

El conjunto tiene 398 autos y 9 columnas. Solo `horsepower` tiene faltantes (6 autos) y no hay filas duplicadas. Por origen hay 249 autos de usa, 79 de japan y 70 de europe. La mayoría tiene 4 cilindros (204), luego 8 (103) y 6 (84); los de 3 y 5 cilindros son 4 y 3. Hay 305 nombres distintos en 398 filas, así que `name` no sirve como variable de entrada. Los 6 autos sin `horsepower` son autos comunes de años entre 1971 y 1982, y no se ve un patrón que explique por qué faltan.

In [ ]:
df.hist(bins=30, figsize=(14, 8))
plt.suptitle("Distribución de cada variable numérica")
plt.tight_layout()
plt.show()

df[["mpg", "displacement", "horsepower", "weight", "acceleration"]].skew().round(2)

In [ ]:
columnas_num = ["mpg", "cylinders", "displacement", "horsepower", "weight", "acceleration", "model_year"]

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df[columnas_num].corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax)
ax.set_title("Correlación entre variables")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
sns.scatterplot(data=df, x="weight", y="mpg", hue="origin", alpha=0.7, ax=axes[0, 0])
axes[0, 0].set_title("mpg contra peso")
sns.scatterplot(data=df, x="horsepower", y="mpg", hue="origin", alpha=0.7, ax=axes[0, 1])
axes[0, 1].set_title("mpg contra caballos de fuerza")
sns.scatterplot(data=df, x="model_year", y="mpg", hue="origin", alpha=0.7, ax=axes[1, 0])
axes[1, 0].set_title("mpg contra año del modelo")
sns.boxplot(data=df, x="origin", y="mpg", hue="origin", ax=axes[1, 1])
axes[1, 1].set_title("mpg por región de origen")
plt.tight_layout()
plt.show()

print("Mediana de mpg por origen:")
print(df.groupby("origin")["mpg"].median())

- `cylinders`, `displacement`, `horsepower` y `weight` están muy correlacionadas entre sí (de 0.84 a 0.95) y todas con `mpg` (de -0.78 a -0.83). Eso se llama multicolinealidad: varias variables dicen casi lo mismo, y en una regresión múltiple sus coeficientes son difíciles de interpretar por separado.
- `weight` es la más correlacionada con `mpg` (-0.83), por eso la usamos en la regresión simple.
- `displacement`, `horsepower` y `weight` tienen cola a la derecha (asimetría de 0.72, 1.09 y 0.53), y su relación con `mpg` se ve curva: el rendimiento cae rápido en los autos ligeros y más lento en los pesados.
- `mpg` sube con el año del modelo (0.58). Los autos de usa tienen peor rendimiento (mediana de 18.5 mpg) que los de europe (26.5) y japan (31.6). Con estos datos no podemos separar cuánto de esa diferencia viene del origen y cuánto de otras variables, como el tamaño del motor; el modelo múltiple ayuda a separarlo.

## 2. Train y test

En regresión no se usa `stratify`, y para la validación cruzada usamos `KFold`. Separamos antes de decidir nada que aprenda de los datos (qué transformar, qué recortar, qué variables conservar). El test no se toca hasta la sección 9.

Agregamos una variable candidata, `potencia_por_peso = horsepower / weight * 1000` (caballos por cada 1000 libras). Se calcula fila por fila y no aprende nada de los datos, así que crearla antes de separar no filtra información. En la sección 3 comprobamos si sirve.

In [ ]:
df["potencia_por_peso"] = df["horsepower"] / df["weight"] * 1000

X = df.drop(columns=["mpg", "name"])    # name no sirve: casi cada auto tiene un nombre distinto
y = df["mpg"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

# Validación cruzada que usamos para comparar: 5 partes, y la misma repetida 10 veces para decisiones finas
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
cv_repetida = RepeatedKFold(n_splits=5, n_repeats=10, random_state=SEED)

print("Train:", X_train.shape, "| Test:", X_test.shape)

## 3. Decisiones de preprocesamiento

Decisiones que ya podemos tomar sin probar nada:

- `name` se quita (casi cada auto tiene un nombre distinto).
- `horsepower` tiene 6 faltantes. Se rellenan con la mediana dentro del Pipeline, para que cada parte de la validación cruzada calcule su propia mediana solo con sus datos de entrenamiento.
- `origin` es nominal: one-hot con `drop="first"`, porque en un modelo lineal una columna redundante causa problemas (lo vimos en clase con `drop_first=True`).
- Todo se estandariza (`StandardScaler`), para que los coeficientes se puedan comparar entre sí y para que Ridge y Lasso penalicen igual a todas las variables.

Decisiones que probamos con validación cruzada, solo con train:

1. Aplicar `log1p` a `displacement`, `horsepower` y `weight`, que tienen cola a la derecha.
2. Recortar los valores extremos (*capping* con el IQR, como en clase).
3. Agregar `potencia_por_peso`.

In [ ]:
numericas = ["cylinders", "displacement", "horsepower", "weight", "acceleration", "model_year"]

q1 = X_train[numericas].quantile(0.25)
q3 = X_train[numericas].quantile(0.75)
iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

es_outlier = (X_train[numericas] < limite_inferior) | (X_train[numericas] > limite_superior)
print("Valores fuera del IQR en train:")
print(es_outlier.sum())

# Copia de train con los extremos recortados a los límites de train (solo para comparar)
X_train_recortado = X_train.copy()
X_train_recortado[numericas] = X_train[numericas].clip(lower=limite_inferior, upper=limite_superior, axis=1)

In [ ]:
def armar_preprocesador(log=True, extra=False):
    asimetricas = ["displacement", "horsepower", "weight"]
    resto = ["cylinders", "acceleration", "model_year"] + (["potencia_por_peso"] if extra else [])
    escalar = [("imputador", SimpleImputer(strategy="median")), ("escalar", StandardScaler())]

    if log:
        pasos = [
            ("asim", Pipeline([("imputador", SimpleImputer(strategy="median")),
                               ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
                               ("escalar", StandardScaler())]), asimetricas),
            ("num", Pipeline(escalar), resto),
        ]
    else:
        pasos = [("num", Pipeline(escalar), asimetricas + resto)]
    pasos.append(("nom", OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False), ["origin"]))
    return ColumnTransformer(pasos)


variantes = {
    "A: sin log ni recorte": dict(log=False, recorte=False, extra=False),
    "B: con log": dict(log=True, recorte=False, extra=False),
    "C: con recorte": dict(log=False, recorte=True, extra=False),
    "D: con log y recorte": dict(log=True, recorte=True, extra=False),
    "E: con log y potencia_por_peso": dict(log=True, recorte=False, extra=True),
}

filas = []
for nombre, opciones in variantes.items():
    datos = X_train_recortado if opciones["recorte"] else X_train
    fila = {"variante": nombre}
    for nombre_modelo, regresion in [("Lineal múltiple", LinearRegression()), ("Polinomial grado 2 + Ridge", None)]:
        if regresion is None:
            modelo = Pipeline([
                ("preprocesador", armar_preprocesador(opciones["log"], opciones["extra"])),
                ("poli", PolynomialFeatures(degree=2, include_bias=False)),
                ("escalador_poli", StandardScaler()),
                ("regresion", Ridge(alpha=30)),
            ])
        else:
            modelo = Pipeline([("preprocesador", armar_preprocesador(opciones["log"], opciones["extra"])),
                               ("regresion", regresion)])
        scores = cross_val_score(modelo, datos, y_train, cv=cv_repetida, scoring="neg_root_mean_squared_error")
        fila[f"RMSE {nombre_modelo}"] = f"{-scores.mean():.3f} ± {scores.std():.3f}"
    filas.append(fila)

pd.DataFrame(filas).set_index("variante")

Cada celda es el RMSE medio ± la desviación entre las 50 evaluaciones (validación cruzada de 5 partes repetida 10 veces). Lo hacemos con dos modelos distintos para ver si la decisión depende del modelo.

- **`log1p` ayuda.** En el modelo lineal baja el RMSE de 3.461 a 3.215 (A contra B), un cuarto de mpg. En el polinomial con Ridge la mejora es pequeña (3.002 a 2.967). Tiene sentido: con la cola a la derecha, unos pocos autos muy grandes dominan la recta, y el logaritmo acerca esos valores al resto.
- **Recortar los extremos no cambia nada.** C y D casi no se mueven frente a A y B (3.453 contra 3.461, y 3.198 contra 3.215), y en el polinomial el recorte incluso empeora un poco (2.975 contra 2.967). Los valores marcados por el IQR son 10 de `horsepower` y 5 de `acceleration`, autos reales y no errores. No recortamos: el Pipeline queda más simple y no se filtra información entre las partes de la validación.
- **`potencia_por_peso` no aporta.** E da 3.239 y 2.981, un poco peor que B (3.215 y 2.967). La información ya está en `horsepower` y `weight`.

Decisión: usamos la variante B, con `log1p` en `displacement`, `horsepower` y `weight`, sin recorte y sin la variable nueva.

In [ ]:
preprocesador = armar_preprocesador(log=True, extra=False)

X_train_proc = preprocesador.fit_transform(X_train)
print("Columnas que ven los modelos:", list(X_train_proc.columns))
print("Faltantes:", X_train_proc.isna().sum().sum())
X_train_proc.describe().loc[["mean", "std"]].round(2)

Quedan 8 columnas sin faltantes. Las columnas numéricas tienen media 0 y desviación 1 en train. `origin` quedó en `nom__origin_japan` y `nom__origin_usa`: europe es la categoría de referencia que se quitó, así que el coeficiente de cada una se lee como la diferencia contra un auto de Europa.

## 4. Cómo ajusta una regresión lineal por dentro

Antes de usar `LinearRegression` calculamos a mano lo que hace y comparamos con scikit-learn.

### 4.1 Regresión simple: la recta que minimiza el error

Con una sola variable, OLS (mínimos cuadrados) elige la recta $\hat{y} = \beta_0 + \beta_1 x$ que hace mínima la suma de los residuos al cuadrado, $\sum (y_i - \hat{y}_i)^2$. La solución tiene fórmula cerrada: $\beta_1 = \dfrac{\text{cov}(x, y)}{\text{var}(x)}$ y $\beta_0 = \bar{y} - \beta_1 \bar{x}$.

In [ ]:
x = X_train["weight"].to_numpy()
y_np = y_train.to_numpy()

beta_1 = np.cov(x, y_np, ddof=0)[0, 1] / np.var(x)
beta_0 = y_np.mean() - beta_1 * x.mean()

modelo_simple = LinearRegression().fit(X_train[["weight"]], y_train)

print(f"A mano:       β0 = {beta_0:.4f} | β1 = {beta_1:.6f}")
print(f"scikit-learn: β0 = {modelo_simple.intercept_:.4f} | β1 = {modelo_simple.coef_[0]:.6f}")
print(f"Por cada 1000 libras más, el rendimiento cambia {beta_1 * 1000:.2f} mpg")

In [ ]:
# Suma de residuos al cuadrado para distintas pendientes (con el mejor intercepto para cada una)
pendientes = np.linspace(beta_1 * 1.6, beta_1 * 0.4, 200)
suma_cuadrados = [((y_np - ((y_np.mean() - b * x.mean()) + b * x)) ** 2).sum() for b in pendientes]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(pendientes, suma_cuadrados)
axes[0].axvline(beta_1, color="tab:red", linestyle="--", label=f"β1 de OLS = {beta_1:.5f}")
axes[0].set_xlabel("Pendiente β1 probada")
axes[0].set_ylabel("Suma de residuos al cuadrado")
axes[0].set_title("El error es mínimo justo en la pendiente de OLS")
axes[0].legend()

recta_x = np.linspace(x.min(), x.max(), 100)
axes[1].scatter(x, y_np, alpha=0.5, s=15)
axes[1].plot(recta_x, beta_0 + beta_1 * recta_x, color="tab:red", linewidth=2.5)
axes[1].set_xlabel("weight")
axes[1].set_ylabel("mpg")
axes[1].set_title("Regresión simple en train")
plt.tight_layout()
plt.show()

La fórmula cerrada da el mismo intercepto y la misma pendiente que scikit-learn: β0 = 46.78 y β1 = -0.007805. Por cada 1000 libras más, el rendimiento baja unas 7.8 mpg.

En la gráfica de la izquierda, cada punto de la curva es una pendiente probada. La suma de residuos al cuadrado forma una curva con un mínimo justo en la pendiente de OLS: eso es lo que significa "entrenar" aquí. En la de la derecha se ve que la nube de puntos es curva y la recta no la alcanza: sobreestima a los autos más pesados y subestima a los intermedios.

### 4.2 Regresión múltiple: la ecuación normal

Con varias variables, el mismo criterio tiene solución en forma de matrices: $\boldsymbol{\beta} = (X^T X)^{-1} X^T y$, donde $X$ lleva una columna de unos para el intercepto. La calculamos con `numpy` sobre los datos ya preprocesados de train y comparamos con `LinearRegression`.

In [ ]:
A = np.column_stack([np.ones(len(X_train_proc)), X_train_proc.to_numpy()])   # columna de unos + variables
beta = np.linalg.solve(A.T @ A, A.T @ y_np)

modelo_ols = LinearRegression().fit(X_train_proc, y_train)

comparacion = pd.DataFrame({
    "a mano": beta[1:],
    "scikit-learn": modelo_ols.coef_,
}, index=X_train_proc.columns)
comparacion.loc["intercepto"] = [beta[0], modelo_ols.intercept_]
display(comparacion.round(4))
print("Diferencia máxima entre los dos:", np.abs(comparacion["a mano"] - comparacion["scikit-learn"]).max())

residuos = y_np - A @ beta
r2_a_mano = 1 - (residuos ** 2).sum() / ((y_np - y_np.mean()) ** 2).sum()
print(f"R² en train a mano: {r2_a_mano:.4f} | con scikit-learn: {modelo_ols.score(X_train_proc, y_train):.4f}")

La ecuación normal da los mismos coeficientes que scikit-learn, con una diferencia de 1.8e-13 (error de redondeo de la computadora), y el mismo R² en train (0.8446). `LinearRegression` no hace nada misterioso: resuelve este sistema de ecuaciones. El intercepto (24.77) es el mpg de un auto de Europa con todas las demás variables en su media.

### 4.3 Multicolinealidad: el factor de inflación de la varianza (VIF)

Cuando una variable se puede predecir bien con las demás, sus coeficientes se vuelven inestables. El VIF de una variable es $1 / (1 - R^2_j)$, donde $R^2_j$ sale de regresar esa variable contra todas las demás. Una regla común dice que un VIF mayor a 10 es una señal fuerte.

In [ ]:
vif = {}
for columna in X_train_proc.columns:
    otras = X_train_proc.drop(columns=columna)
    r2_columna = LinearRegression().fit(otras, X_train_proc[columna]).score(otras, X_train_proc[columna])
    vif[columna] = 1 / (1 - r2_columna)

pd.Series(vif, name="VIF").sort_values(ascending=False).round(1)

Cuatro variables están en o cerca de la zona de alerta: `displacement` (26.1), `weight` (15.8), `horsepower` (10.4) y `cylinders` (9.8). Cada una se puede predecir muy bien con las demás, así que el modelo no puede repartir con precisión cuánto le toca a cada una. Por eso sus coeficientes son inestables y pueden confundir. Un ejemplo es `cylinders`: su coeficiente es positivo (+1.11) aunque su correlación con `mpg` es negativa (-0.78), porque está corrigiendo a `displacement`, `horsepower` y `weight`. Esto no daña las predicciones, pero sí la lectura de los coeficientes, y es una razón para probar Ridge y Lasso.

### 4.4 Ridge y Lasso: qué le hace la penalización a los coeficientes

- **Ridge** minimiza el error más $\alpha \sum \beta_j^2$. También tiene solución cerrada: $\boldsymbol{\beta} = (X^T X + \alpha I)^{-1} X^T y$ (con los datos centrados, sin penalizar el intercepto). Encoge los coeficientes sin llegar a cero.
- **Lasso** minimiza el error más $\alpha \sum |\beta_j|$. No tiene fórmula cerrada, pero con la penalización algunos coeficientes llegan exactamente a cero.

Comprobamos la fórmula de Ridge y dibujamos cómo cambian los coeficientes al subir $\alpha$.

In [ ]:
X_centrada = X_train_proc.to_numpy() - X_train_proc.to_numpy().mean(axis=0)
y_centrada = y_np - y_np.mean()

for alfa in [1, 30]:
    beta_ridge = np.linalg.solve(X_centrada.T @ X_centrada + alfa * np.eye(X_centrada.shape[1]), X_centrada.T @ y_centrada)
    ridge = Ridge(alpha=alfa).fit(X_train_proc, y_train)
    print(f"Ridge con alpha = {alfa}: diferencia máxima entre la fórmula y scikit-learn = {np.abs(beta_ridge - ridge.coef_).max():.1e}")

alfas_ridge = np.logspace(-2, 3, 40)
alfas_lasso = np.logspace(-3, 0.5, 40)
camino_ridge = pd.DataFrame([Ridge(alpha=a).fit(X_train_proc, y_train).coef_ for a in alfas_ridge],
                            index=alfas_ridge, columns=X_train_proc.columns)
camino_lasso = pd.DataFrame([Lasso(alpha=a).fit(X_train_proc, y_train).coef_ for a in alfas_lasso],
                            index=alfas_lasso, columns=X_train_proc.columns)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for camino, ax, titulo in [(camino_ridge, axes[0], "Ridge"), (camino_lasso, axes[1], "Lasso")]:
    for columna in camino.columns:
        ax.plot(camino.index, camino[columna], label=columna)
    ax.set_xscale("log")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(f"Coeficientes de {titulo} según alpha")
    ax.set_xlabel("alpha (escala logarítmica)")
    ax.set_ylabel("Coeficiente")
axes[1].legend(loc="center left", bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.show()

print("Variables que Lasso deja en cero:")
for alfa in [0.01, 0.1, 0.5]:
    coef = Lasso(alpha=alfa).fit(X_train_proc, y_train).coef_
    print(f"  alpha = {alfa}:", [c for c, v in zip(X_train_proc.columns, coef) if v == 0] or "ninguna")

La fórmula de Ridge coincide con scikit-learn (diferencias de 1e-14). En la gráfica de la izquierda, al subir `alpha` todos los coeficientes se encogen poco a poco y ninguno llega a cero. En la de la derecha, Lasso sí los lleva a cero: con `alpha=0.1` deja en cero `displacement`, `cylinders` y `origin_japan`, y con `alpha=0.5` también `acceleration` y `origin_usa`, de modo que solo quedan `displacement`, `horsepower`, `weight` y `model_year`. Con `alpha=0.01`, el valor de la clase, Lasso no elimina ninguna: penaliza muy poco.

### 4.5 RANSAC: ajustar con los autos "normales"

`RANSACRegressor` ajusta la regresión muchas veces con subconjuntos al azar y se queda con la que deja más puntos cerca de la recta (los *inliers*). Un punto es inlier si su residuo es menor que `residual_threshold`; si no se indica, usa la mediana de las desviaciones absolutas de `y` (MAD). Los demás puntos no participan en el ajuste final.

In [ ]:
ransac = RANSACRegressor(LinearRegression(), random_state=SEED).fit(X_train_proc, y_train)
mad = np.median(np.abs(y_np - np.median(y_np)))
es_inlier = ransac.inlier_mask_

print(f"Umbral por defecto (MAD de mpg): {mad:.2f} mpg")
print(f"Autos que usa para el ajuste: {es_inlier.sum()} de {len(es_inlier)} ({es_inlier.mean() * 100:.1f}%)")

fuera = pd.DataFrame({"origin": X_train["origin"].to_numpy(), "inlier": es_inlier})
print("\nPorcentaje de autos que RANSAC deja fuera, por origen:")
print(((1 - fuera.groupby("origin")["inlier"].mean()) * 100).round(1))

Con su umbral por defecto (6.30 mpg), RANSAC ajusta con 270 de los 318 autos (84.9%) y deja fuera 48. Los deja fuera de forma bastante pareja por origen (de 14% a 18%), así que no parecen errores del conjunto, sino autos que una recta no alcanza bien. Para RANSAC eso le cuesta: pierde datos útiles.

## 5. Modelo lineal base

El modelo base es la regresión lineal múltiple con todas las variables ya preprocesadas. Aquí vemos sus coeficientes y sus métricas en train y en validación cruzada. Las métricas en test llegan hasta la sección 9.

In [ ]:
def metricas_regresion(y_real, y_pred, n_variables):
    n = len(y_real)
    r2 = r2_score(y_real, y_pred)
    return {
        "MSE": mean_squared_error(y_real, y_pred),
        "RMSE": root_mean_squared_error(y_real, y_pred),
        "MAE": mean_absolute_error(y_real, y_pred),
        "MAPE": mean_absolute_percentage_error(y_real, y_pred),
        "R²": r2,
        "R² ajustado": 1 - (1 - r2) * (n - 1) / (n - n_variables - 1),
    }


modelo_base = Pipeline([("preprocesador", armar_preprocesador()), ("regresion", LinearRegression())]).fit(X_train, y_train)
coef_base = modelo_base.named_steps["regresion"]

print("Intercepto:", round(coef_base.intercept_, 3), "(mpg de un auto de Europa con las demás variables en su media)")
coeficientes = pd.DataFrame({
    "variable": X_train_proc.columns,
    "coeficiente": coef_base.coef_,
}).sort_values("coeficiente", key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=coeficientes, x="coeficiente", y="variable", hue="variable", palette="coolwarm", legend=False, ax=ax)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Coeficientes de la regresión múltiple (variables estandarizadas)")
plt.show()

display(coeficientes.round(3))

pred_train_base = modelo_base.predict(X_train)
cv_base = cross_validate(Pipeline([("preprocesador", armar_preprocesador()), ("regresion", LinearRegression())]),
                         X_train, y_train, cv=cv,
                         scoring={"rmse": "neg_root_mean_squared_error", "mae": "neg_mean_absolute_error", "r2": "r2"})
resumen_base = pd.DataFrame({
    "train": metricas_regresion(y_train, pred_train_base, n_variables=X_train_proc.shape[1]),
}).loc[["MSE", "RMSE", "MAE", "MAPE", "R²", "R² ajustado"]]
resumen_base["validación cruzada"] = [np.nan, -cv_base["test_rmse"].mean(), -cv_base["test_mae"].mean(), np.nan,
                                      cv_base["test_r2"].mean(), np.nan]
resumen_base.round(3)

In [ ]:
residuos_base = y_train - pred_train_base

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].scatter(y_train, pred_train_base, alpha=0.6, s=15)
axes[0].plot([10, 45], [10, 45], "k--", label="Predicción perfecta")
axes[0].set_xlabel("mpg real")
axes[0].set_ylabel("mpg predicho")
axes[0].set_title("Real contra predicho (train)")
axes[0].legend()

sns.histplot(residuos_base, bins=25, ax=axes[1])
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set_xlabel("Residuo (real - predicho)")
axes[1].set_title("Distribución de los residuos")

axes[2].scatter(pred_train_base, residuos_base, alpha=0.6, s=15)
axes[2].axhline(0, color="black", linestyle="--")
axes[2].set_xlabel("mpg predicho")
axes[2].set_ylabel("Residuo")
axes[2].set_title("Residuos contra predicho")
plt.tight_layout()
plt.show()

El modelo base tiene un R² de 0.845 en train y 0.832 en validación cruzada, y un RMSE de 3.12 y 3.21 mpg. La diferencia entre train y validación es pequeña, así que no hay un sobreajuste fuerte. En train, el MAE es de 2.35 mpg y el MAPE de 10.5%.

Los coeficientes están en mpg por cada desviación estándar de la variable (para `origin`, contra Europa):

- `weight` es lo que más pesa (-4.53): a más peso, menos rendimiento.
- `model_year` es el segundo (+2.91): los autos más nuevos rinden más.
- `origin_usa` (-1.79): con lo demás igual, un auto de usa rinde 1.8 mpg menos que uno de Europa.
- `horsepower` (-1.56) y, con menos peso, `displacement` (-0.60) y `acceleration` (-0.40).
- `cylinders` sale positivo (+1.11) por la multicolinealidad que vimos en la sección 4.3, así que no se puede leer solo.

Los residuos están centrados en cero y se parecen a una campana, pero la gráfica de residuos contra predicho tiene forma de curva: son positivos cuando el predicho es muy bajo, negativos en el centro y se abren hacia arriba en los autos de más rendimiento. Es la señal de que la relación no es lineal, y una recta no la captura.

## 6. Los modelos de la clase

Probamos los seis modelos con los hiperparámetros que se usaron en la lección: Ridge con `alpha=1`, Lasso con `alpha=0.01`, polinomial de grado 2 y RANSAC con su configuración por defecto. La función `evaluar_cv` calcula RMSE, MAE y $R^2$ con validación cruzada en train, para cada modelo.

In [ ]:
TODAS = list(X_train.columns)


def evaluar_cv(modelos, validacion=cv):
    filas = []
    for nombre, (modelo, columnas) in modelos.items():
        r = cross_validate(modelo, X_train[columnas], y_train, cv=validacion, return_train_score=True,
                           scoring={"rmse": "neg_root_mean_squared_error", "mae": "neg_mean_absolute_error", "r2": "r2"})
        filas.append({
            "modelo": nombre,
            "RMSE train": -r["train_rmse"].mean(),
            "RMSE validación": -r["test_rmse"].mean(),
            "desviación": r["test_rmse"].std(),
            "MAE validación": -r["test_mae"].mean(),
            "R² validación": r["test_r2"].mean(),
        })
    return pd.DataFrame(filas).set_index("modelo")


def con_polinomio(regresion, grado=2):
    return Pipeline([
        ("preprocesador", armar_preprocesador()),
        ("poli", PolynomialFeatures(degree=grado, include_bias=False)),
        ("escalador_poli", StandardScaler()),
        ("regresion", regresion),
    ])


def lineal(regresion):
    return Pipeline([("preprocesador", armar_preprocesador()), ("regresion", regresion)])


modelos_clase = {
    "Simple (weight)": (LinearRegression(), ["weight"]),
    "Lineal múltiple": (lineal(LinearRegression()), TODAS),
    "Ridge (alpha=1)": (lineal(Ridge(alpha=1.0)), TODAS),
    "Lasso (alpha=0.01)": (lineal(Lasso(alpha=0.01)), TODAS),
    "Polinomial (grado 2)": (con_polinomio(LinearRegression()), TODAS),
    "RANSAC": (lineal(RANSACRegressor(LinearRegression(), random_state=SEED)), TODAS),
}

tabla_clase = evaluar_cv(modelos_clase)
tabla_clase.round(3)

- **Simple (`weight`)**: RMSE de 4.49. Una sola variable no alcanza.
- **Lineal múltiple**: baja a 3.21, más de un mpg menos de error. Es el modelo base.
- **Ridge y Lasso**: 3.20 los dos, igual que el lineal. Con 8 variables y 318 autos no hay mucho que frenar, y los dos valores de `alpha` son pequeños.
- **Polinomial de grado 2**: es el mejor hasta aquí (3.07), así que sí hay curvatura. Pero su RMSE en train es 2.44 contra 3.07 en validación: el hueco es de 0.63, contra 0.10 del lineal. Crea 44 columnas con solo 318 autos, y empieza a sobreajustar. Su desviación entre partes también es la más alta (0.248).
- **RANSAC**: el peor de los modelos múltiples (3.48), porque descarta autos reales.

## 7. Ajuste de hiperparámetros

Con `GridSearchCV` buscamos el mejor `alpha` en Ridge y Lasso, el mejor grado del polinomial y el mejor `residual_threshold` en RANSAC, siempre con validación cruzada en train. También probamos un polinomial con Ridge: el polinomial de grado 2 crea 44 columnas con solo 318 autos en train, así que puede sobreajustar, y Ridge lo frena. El test no se usa en la búsqueda.

In [ ]:
busquedas = {
    "Ridge": (lineal(Ridge()), {"regresion__alpha": [0.01, 0.1, 1, 10, 100, 300]}),
    "Lasso": (lineal(Lasso()), {"regresion__alpha": [0.0005, 0.001, 0.01, 0.1, 1]}),
    "Polinomial": (con_polinomio(LinearRegression()), {"poli__degree": [1, 2, 3]}),
    "Polinomial + Ridge": (con_polinomio(Ridge()), {"poli__degree": [2, 3, 4],
                                                    "regresion__alpha": [1, 10, 30, 100, 300, 1000]}),
    "RANSAC": (lineal(RANSACRegressor(LinearRegression(), random_state=SEED)),
               {"regresion__residual_threshold": [1, 2, 3, 4, 5, 8, 12, 20]}),
}

ajustados = {}
filas = []
for nombre, (modelo, rejilla) in busquedas.items():
    busqueda = GridSearchCV(modelo, rejilla, cv=cv, scoring="neg_root_mean_squared_error").fit(X_train, y_train)
    ajustados[nombre] = busqueda.best_estimator_
    filas.append({"modelo": nombre, "mejores hiperparámetros": busqueda.best_params_,
                  "RMSE validación (5 partes)": -busqueda.best_score_})

pd.DataFrame(filas).set_index("modelo").round(3)

- Ridge, Lasso y el polinomial: la búsqueda eligió los mismos valores de la clase (`alpha=1`, `alpha=0.01` y grado 2). Por eso, en la tabla de la sección 8, cada uno aparece dos veces con resultados iguales.
- Polinomial con Ridge: el mejor fue grado 3 con `alpha=30`, con RMSE de 2.901. Es el que más baja el error: el polinomio capta la curvatura y Ridge frena los coeficientes de las muchas columnas nuevas (de 8 variables salen 164 columnas en grado 3).
- RANSAC: el mejor `residual_threshold` fue 12 mpg, un umbral grande con el que casi todos los autos son inliers y se parece a una regresión lineal normal. Queda en 3.209, igual que el lineal, así que RANSAC no aporta aquí.

## 8. Elegir el modelo final

Juntamos los modelos de la clase con los ajustados y los medimos con la validación cruzada repetida (5 partes, 10 veces), que da una comparación más estable. La regla se escribe antes de mirar el resultado: si varios modelos quedan a menos de 0.05 mpg de RMSE del mejor, nos quedamos con el más simple, entendiendo por simple el que menos columnas y menos transformaciones usa.

In [ ]:
modelos_finales = dict(modelos_clase)
modelos_finales["Ridge (ajustado)"] = (ajustados["Ridge"], TODAS)
modelos_finales["Lasso (ajustado)"] = (ajustados["Lasso"], TODAS)
modelos_finales["Polinomial (ajustado)"] = (ajustados["Polinomial"], TODAS)
modelos_finales["Polinomial + Ridge"] = (ajustados["Polinomial + Ridge"], TODAS)
modelos_finales["RANSAC (ajustado)"] = (ajustados["RANSAC"], TODAS)

# Orden de simplicidad: 1 es lo más simple
simplicidad = {
    "Simple (weight)": 1, "Lineal múltiple": 2, "Ridge (alpha=1)": 3, "Ridge (ajustado)": 3,
    "Lasso (alpha=0.01)": 3, "Lasso (ajustado)": 3, "RANSAC": 4, "RANSAC (ajustado)": 4,
    "Polinomial (grado 2)": 5, "Polinomial (ajustado)": 5, "Polinomial + Ridge": 6,
}

tabla_cv = evaluar_cv(modelos_finales, validacion=cv_repetida)
tabla_cv["simplicidad"] = pd.Series(simplicidad)
tabla_cv = tabla_cv.sort_values("RMSE validación")
display(tabla_cv.round(3))

MARGEN = 0.05
mejor_rmse = tabla_cv["RMSE validación"].min()
aceptables = tabla_cv[tabla_cv["RMSE validación"] <= mejor_rmse + MARGEN]
nombre_final = aceptables.sort_values(["simplicidad", "RMSE validación"]).index[0]

print("Modelos a menos de", MARGEN, "mpg del mejor:", list(aceptables.index))
print("Modelo final:", nombre_final)

El polinomial con Ridge (grado 3 y `alpha=30`) tiene el RMSE de validación más bajo, 2.901, y es el único a menos de 0.05 mpg del mejor: el siguiente, el polinomial de grado 2, queda en 3.058. Por la regla que escribimos antes, el modelo final es el **polinomial con Ridge**.

Dos cosas para tener presentes. La primera es el costo: con grado 3 el modelo usa 164 columnas con solo 318 autos de entrenamiento, y su RMSE en train (2.341) es menor que el de validación (2.901). Ridge frena el sobreajuste, pero no lo elimina. La segunda es que los coeficientes de un polinomio ya no se pueden interpretar. Si lo importante fuera explicar qué variables mueven el rendimiento, el lineal múltiple (3.215) sería la mejor opción, a cambio de unos 0.3 mpg más de error.

## 9. Tabla final con test

Esta es la primera vez que usamos el test. La decisión del modelo final ya está tomada, así que estos números no la cambian.

In [ ]:
filas = []
predicciones = {}
for nombre, (modelo, columnas) in modelos_finales.items():
    modelo.fit(X_train[columnas], y_train)
    pred = modelo.predict(X_test[columnas])
    predicciones[nombre] = pred
    m = metricas_regresion(y_test, pred, n_variables=len(columnas))
    filas.append({"modelo": nombre, "RMSE test": m["RMSE"], "MAE test": m["MAE"], "MAPE test": m["MAPE"], "R² test": m["R²"]})

tabla_test = pd.DataFrame(filas).set_index("modelo")
reporte_final = tabla_cv[["RMSE train", "RMSE validación", "desviación"]].join(tabla_test).round(3)
reporte_final

In [ ]:
largo = (reporte_final[["RMSE train", "RMSE validación", "RMSE test"]]
         .reset_index()
         .melt(id_vars="modelo", var_name="conjunto", value_name="RMSE"))

fig, ax = plt.subplots(figsize=(9, 7))
sns.barplot(data=largo, x="RMSE", y="modelo", hue="conjunto", palette="Set2", ax=ax)
ax.set_title("Comparación de modelos (más bajo es mejor)")
ax.set_xlabel("RMSE (mpg)")
ax.set_ylabel("")
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
plt.show()

El orden en test es casi el mismo que en validación. El polinomial con Ridge tiene RMSE de 2.176, MAE de 1.642 y R² de 0.912, y se equivoca en promedio un 7.6% (MAPE). El lineal múltiple saca 2.726 y 0.862, y la regresión simple 3.859 y 0.723. RANSAC ajustado (2.667) queda parecido a los lineales (de 2.70 a 2.73): la diferencia es de unas décimas de mpg y está dentro del ruido de un test de 80 autos.

Hay un detalle que conviene mirar: el RMSE de test del modelo final (2.176) es menor que el de validación (2.901). Con un test tan chico eso puede pasar por suerte, y la sección 10 lo revisa.

In [ ]:
pred_final = predicciones[nombre_final]
residuos_final = y_test - pred_final

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].scatter(y_test, pred_final, alpha=0.7, s=18)
axes[0].plot([10, 45], [10, 45], "k--", label="Predicción perfecta")
axes[0].set_xlabel("mpg real")
axes[0].set_ylabel("mpg predicho")
axes[0].set_title(f"{nombre_final}: real contra predicho (test)")
axes[0].legend()

sns.histplot(residuos_final, bins=15, ax=axes[1])
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set_xlabel("Residuo (real - predicho)")
axes[1].set_title("Distribución de los residuos (test)")

axes[2].scatter(pred_final, residuos_final, alpha=0.7, s=18)
axes[2].axhline(0, color="black", linestyle="--")
axes[2].set_xlabel("mpg predicho")
axes[2].set_ylabel("Residuo")
axes[2].set_title("Residuos contra predicho (test)")
plt.tight_layout()
plt.show()

errores = X_test[["origin", "cylinders"]].copy()
errores["residuo"] = residuos_final
por_origen = errores.groupby("origin")["residuo"].agg(autos="size", RMSE=lambda s: np.sqrt((s ** 2).mean()), sesgo="mean")
por_cilindros = errores.groupby("cylinders")["residuo"].agg(autos="size", RMSE=lambda s: np.sqrt((s ** 2).mean()), sesgo="mean")
display(por_origen.round(2))
display(por_cilindros.round(2))

Los puntos quedan cerca de la diagonal y los residuos están centrados en cero. Los errores más grandes son de unos 7 mpg y son sobreestimaciones. Por grupos:

- **Origen:** europe tiene el RMSE más alto (2.78) y un sesgo de -1.06: el modelo predice en promedio 1 mpg de más para esos 14 autos. En japan el RMSE es 2.23 y en usa 1.97, casi sin sesgo.
- **Cilindros:** los de 6 cilindros tienen el RMSE más alto (2.53, con 14 autos), después los de 4 (2.33, con 42) y los de 8 el más bajo (1.60, con 24).

Los grupos tienen entre 13 y 53 autos, así que estas diferencias no son concluyentes.

## 10. ¿Qué tan estable es el resultado?

El test tiene solo 80 autos. Para ver cuánto depende el resultado de qué autos cayeron en test, repetimos el proceso con 20 repartos distintos (`random_state` de 0 a 19). En cada uno separamos, ajustamos y medimos en su test: la regresión simple, la lineal múltiple y el polinomial con Ridge (eligiendo su grado y su `alpha` solo con el train de ese reparto). Esto no sirve para escoger el modelo; solo mide la variación.

In [ ]:
rejilla_chica = {"poli__degree": [2, 3], "regresion__alpha": [10, 30, 100, 300]}

filas = []
for semilla in list(range(20)) + [SEED]:
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=semilla)

    simple = LinearRegression().fit(X_tr[["weight"]], y_tr)
    multiple = lineal(LinearRegression()).fit(X_tr, y_tr)
    busqueda_reparto = GridSearchCV(con_polinomio(Ridge()), rejilla_chica, cv=cv,
                                    scoring="neg_root_mean_squared_error").fit(X_tr, y_tr)

    filas.append({
        "reparto": semilla,
        "Simple (weight)": root_mean_squared_error(y_te, simple.predict(X_te[["weight"]])),
        "Lineal múltiple": root_mean_squared_error(y_te, multiple.predict(X_te)),
        "Polinomial + Ridge": root_mean_squared_error(y_te, busqueda_reparto.predict(X_te)),
    })
repartos = pd.DataFrame(filas).set_index("reparto")

otros = repartos.drop(index=SEED)    # los 20 repartos distintos al del notebook
resumen = pd.DataFrame({
    "media": otros.mean(), "mínimo": otros.min(), "máximo": otros.max(),
    "reparto 42": repartos.loc[SEED],
})
display(resumen.round(3))
print("Repartos en los que el polinomial con Ridge tiene menor RMSE que el lineal múltiple:",
      (otros["Polinomial + Ridge"] < otros["Lineal múltiple"]).sum(), "de 20")
print("Diferencia media de RMSE (lineal múltiple menos polinomial con Ridge):",
      round((otros["Lineal múltiple"] - otros["Polinomial + Ridge"]).mean(), 3), "mpg")

In [ ]:
largo = otros.reset_index().melt(id_vars="reparto", var_name="modelo", value_name="RMSE en test")

fig, ax = plt.subplots(figsize=(8, 4))
sns.stripplot(data=largo, x="RMSE en test", y="modelo", color="tab:blue", size=7, alpha=0.7, ax=ax)
for posicion, modelo in enumerate(otros.columns):
    ax.plot(repartos.loc[SEED, modelo], posicion, marker="D", color="tab:red", markersize=9,
            label="Reparto 42" if posicion == 0 else None)
ax.set_title("RMSE en test con 20 repartos distintos (cada punto es un reparto)")
ax.set_xlabel("RMSE en test (mpg)")
ax.set_ylabel("")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

Con 20 repartos distintos, el polinomial con Ridge saca un RMSE medio de 2.846 en test (de 2.452 a 3.913), el lineal múltiple 3.113 (de 2.649 a 3.581) y la regresión simple 4.319. El polinomial tiene menor error que el lineal en 19 de los 20 repartos, con una diferencia media de 0.268 mpg.

Eso corrige la impresión que daba el reparto 42. Ahí el polinomial saca 2.176, por debajo del mínimo de los otros 20 repartos (2.452), y la diferencia contra el lineal parece de 0.55 mpg. Es un reparto favorable para el polinomial. Lo razonable es esperar con autos nuevos un RMSE más cercano a 2.85 que a 2.18. También se ve que el polinomial es el que más varía (llega a 3.913 en un reparto), porque es el más flexible.

## Conclusiones

- **Modelo elegido.** El polinomial de grado 3 con Ridge (`alpha=30`): RMSE de 2.901 mpg en validación cruzada repetida, y 2.176 mpg, MAE de 1.642 y R² de 0.912 en test. Con 20 repartos distintos el RMSE esperado en test es de unos 2.85 mpg, y supera al lineal múltiple en 19 de 20.
- **Lo que más ayudó.** Aplicar `log1p` a `displacement`, `horsepower` y `weight` (el lineal baja de 3.461 a 3.215) y capturar la curvatura con un polinomio (de 3.215 a 2.901).
- **Lo que no ayudó.** Recortar los extremos, la variable `potencia_por_peso`, Ridge y Lasso sobre las variables originales (con 8 variables no hay mucho que frenar) y RANSAC (descarta autos reales).
- **Cómo funciona por dentro.** La ecuación normal da los mismos coeficientes que `LinearRegression`, y la fórmula de Ridge los mismos que `Ridge`. El VIF muestra una multicolinealidad fuerte (`displacement` 26.1, `weight` 15.8, `horsepower` 10.4, `cylinders` 9.8), por lo que esos coeficientes no se leen por separado.
- **Si importara explicar.** El lineal múltiple: error de 2.73 mpg en test contra 2.18 del polinomial, pero con coeficientes que se pueden leer (peso -4.53, año del modelo +2.91, origen usa -1.79).
- **Limitaciones.** El test tiene 80 autos y el reparto 42 resultó favorable al polinomial. Los datos son de autos de 1970 a 1982, así que el modelo no sirve para autos actuales. El polinomio no es interpretable. Elegimos el modelo con validación cruzada sobre el mismo train, así que su 2.901 es algo optimista.
- **Qué probaríamos después.** Validación cruzada anidada para medir todo el proceso de elección, y otros modelos que veremos más adelante.

### Decisiones

| # | Decisión | Opciones | Elección | Evidencia (solo train) |
|---|---|---|---|---|
| 1 | `name` | Conservar o quitar | Quitar | 305 nombres distintos en 398 autos |
| 2 | `horsepower` faltante | Mediana, borrar filas | Mediana dentro del Pipeline | Solo 6 autos y sin patrón visible |
| 3 | `origin` | One-hot con o sin `drop` | One-hot con `drop="first"` | Evita una columna redundante en modelos lineales |
| 4 | `log1p` en `displacement`, `horsepower`, `weight` | Aplicarlo o no | Aplicarlo | RMSE lineal 3.461 contra 3.215 |
| 5 | Recortar extremos (IQR) | Recortar o no | No recortar | 3.215 contra 3.198, diferencia menor que el ruido |
| 6 | `potencia_por_peso` | Agregarla o no | No agregarla | 3.239 contra 3.215 |
| 7 | Hiperparámetros | Los de la clase o `GridSearchCV` | `GridSearchCV` con validación en train | Polinomial con Ridge: grado 3 y `alpha=30`, RMSE 2.901 |
| 8 | Modelo final | Entre los de la tabla | Polinomial con Ridge | Regla: a menos de 0.05 mpg del mejor, el más simple (era el único) |